# Audit Generative LLM Escalations & Diagnostics
This notebook runs the audit and diagnostic pipeline for the **LinkNER Architecture** on Lifestyle Factors (LSF) biomedical abstracts.

---

### 🔍 How Escalations and Risk Scores are Computed

1. **Stage 1: Primary SpanNER Recognition & Uncertainty (U_SpanNER)**
   - SpanNER evaluates candidate token spans up to width 6 across the abstract.
   - For each span, uncertainty is computed using Prediction Entropy (PE), Least Confidence (LC), MCD, ENN, or Margin:
     `U_SpanNER = - Sum of [ P(c) * log(P(c)) ] / log(Num_Classes)`
   - Normalizes uncertainty into [0, 1]. High entropy indicates model indecision.

2. **Stage 2: LOF Vector Novelty Scoring (Novelty)**
   - Candidate spans are embedded into high-dimensional vector representations.
   - Their embeddings are evaluated against training entity exemplars using K-Nearest Neighbor (KNN) cosine distance and scikit-learn Local Outlier Factor (LOF).
   - Combined scores are mapped through a reference-calibrated logistic function into [0, 1]:
     `Novelty = 1.0 / (1.0 + exp( - (Raw_Novelty_Score - mu) / sigma ))`
   - Higher novelty signifies out-of-vocabulary, rare, or atypical biomedical terminology.
   - *Optimization*: When running with precomputed cache, cached novelty scores are reused directly.

3. **Stage 3: Composite Unreliability & Escalation Gating**
   - The unified gating score combines both uncertainty and novelty:
     `Unreliability = (w_novelty * Novelty) + (w_uncertainty * U_SpanNER)`
   - **Gating Rule**:
     - If `Unreliability < Threshold` (e.g. 0.35): Accepted locally by SpanNER (0 API cost).
     - If `Unreliability >= Threshold`: Escalated to the Generative LLM.

4. **Stage 4: Direct Generative LLM Resolution & 10-Outcome Auditing**
   - Escalated spans are bundled into a single structured prompt with Definitions & Guidelines (D&G).
   - The Generative LLM assigns the true entity category (or "None of the above").
   - The LLM decision is directly adopted as the final label (`final_label = llm_label`).
   - Each escalated span is classified into 1 of 10 canonical diagnostic outcome categories:
     1. `GT_X_BERT_Y_LLM_X`: BERT proposed incorrect LSF category, Generator successfully corrected to Ground Truth.
     2. `GT_O_BERT_X_LLM_O`: BERT false alarm on Non-LSF span, Generator correctly dropped to 'O'.
     3. `GT_X_BERT_X_LLM_X`: BERT proposed correct LSF category, Generator preserved it.
     4. `GT_X_BERT_X_LLM_Y`: BERT proposed correct LSF category, Generator mislabeled to different LSF category.
     5. `GT_O_BERT_X_LLM_X`: Both agreed on the same false positive LSF category for a Non-LSF span.
     6. `GT_O_BERT_X_LLM_Y`: Both predicted different incorrect LSF categories for a Non-LSF span.
     7. `GT_X_BERT_X_LLM_O`: BERT proposed correct LSF category, Generator mistakenly discarded to Non-LSF ('O').
     8. `GT_X_BERT_Y_LLM_O`: Both made mistakes: BERT predicted wrong category, Generator discarded to 'O'.
     9. `GT_X_BERT_Y_LLM_Z`: Both predicted different incorrect LSF categories, neither matching Ground Truth.
     10. `GT_X_BERT_Y_LLM_Y`: Both agreed on the same incorrect LSF category, neither matching Ground Truth.


In [1]:
# ==============================================================================
# [CELL 1]: Setup, Environment & Project Path Resolution
# ==============================================================================
import os
import sys
import glob
import json
import importlib

# Set Project Root and add to sys.path
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

# Ensure fresh reload of pipeline module
from src.architectures.hybrid_linkner import pipeline as pipeline_module
importlib.reload(pipeline_module)
from src.architectures.hybrid_linkner.pipeline import LinkNERPipeline
from src.data.dataset_converter import parse_ann_file
from src.common.label_mapping import detect_schema_from_path
from src.data.cached_loader import load_cached_spanner_candidates, get_cache_split_key
from src.common.audit_logger import (
    classify_10_outcome_str,
    classify_audit_outcome,
    get_audit_status,
    compute_canonical_audit_breakdown,
    format_canonical_audit_table,
    format_audit_summary_box,
    CANONICAL_OUTCOME_CATEGORIES,
    AUDIT_CATEGORIES
)


/Users/hmd/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [2]:
# ==============================================================================
# [CELL 2]: Warning Filters & Runtime Configurations
# ==============================================================================
import warnings
warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)


In [ ]:
# ==============================================================================
# [CELL 3]: Dataset & Experiment Configuration (Multi-Dataset Cached Presets)
# ==============================================================================
SEED = int(os.getenv("RANDOM_SEED", "42"))

# 1. Dataset Directory Configuration:
# Standard validation set (80 abstracts) exactly matching the 400/val cache:
#   Preset A (Recommended): data/new/400Abstracts/val -> maps to cache "400/val" (80 abstracts, 1492 spans)
#   Preset B: data/new/400Abstracts/train             -> maps to cache "400/train" (240 abstracts, 3713 spans)
#   Preset C: data/new/200Abstracts/train             -> maps to cache "200/train" (120/160 abstracts, 1570 spans)
DATA_DIR = os.getenv("DATA_DIR", os.path.join(PROJECT_ROOT, "data/new/400Abstracts/val"))
if not os.path.exists(DATA_DIR):
    DATA_DIR = os.path.join(PROJECT_ROOT, "data/new/400Abstracts/no_disease/val")

# Target ontology schema (True = New Canonical Schema)
USE_NEW_LABELS = True

# 2. Pipeline Execution & Gating Parameters (Optimal Preset):
BATCH_SIZE = int(os.getenv("BATCH_SIZE", "5"))
METHOD = "margin"                      # Uncertainty Metric: MARGIN (with MCD)
GATING_MODE = "prob_or"                # Gating Operator: PROB_OR
W_NOVELTY = 0.30                       # w_novelty = 0.30
W_UNCERTAINTY = 0.70                   # w_uncertainty = 0.70
UNRELIABILITY_THRESHOLD = 0.35         # Tau = 0.35
THRESHOLD = UNRELIABILITY_THRESHOLD
NOVELTY_THRESHOLD = 0.60
MARGIN_THRESHOLD = 0.50                # Margin Veto Safeguard = 0.50
USE_MARGIN_SAFEGUARD = True
USE_NMS = False                        # NMS Policy: Disabled (Drop NMS)
# Available Single-Call Prompts:
# - "v1_allowed_new_span" : With BERT label evidence + permission to detect & classify missed LSF spans
# - "v1_with_label"       : Standard single-call with BERT label evidence
# - "v2_no_label"         : Single-call without candidate labels
PROMPT_VERSION = "v1_allowed_new_span"
MCD_PASSES = 5
ACTIVE_METRICS = ["strict", "partial", "mention"]

# 3. Cached SpanNER & Novelty Output Configuration (Zero-Overhead Exact Reproducibility)
USE_CACHED_SPANNER = os.getenv("USE_CACHED_SPANNER", "true").lower() in ("true", "1", "yes")
CACHED_SPANNER_DIR = os.path.join(PROJECT_ROOT, "data/cached/spanner_160_10_class")
CACHE_SPLIT_KEY = get_cache_split_key(DATA_DIR)

schema_name = "NEW (Relabeled)" if USE_NEW_LABELS else "LEGACY"
print("=" * 80)
print(f"  📋 EXPERIMENT CONFIGURATION (OPTIMAL PRESET)")
print("=" * 80)
print(f"  • Data Directory         : {DATA_DIR}")
print(f"  • Cache Split Key        : {CACHE_SPLIT_KEY} (Directory: {CACHED_SPANNER_DIR})")
print(f"  • Use Cached Data        : {USE_CACHED_SPANNER}")
print(f"  • Label Schema           : {schema_name} (USE_NEW_LABELS = {USE_NEW_LABELS})")
print(f"  • Batch Size (B)         : {BATCH_SIZE} abstracts per batch")
print(f"  • Gating Operator        : {GATING_MODE.upper()}")
print(f"  • Weight Combination     : w_novelty={W_NOVELTY:.2f}, w_uncertainty={W_UNCERTAINTY:.2f}")
print(f"  • Uncertainty Metric     : {METHOD.upper()} (MCD passes={MCD_PASSES})")
print(f"  • Novelty Metric         : LOF (Precomputed from cache if available)")
print(f"  • Threshold (Tau)        : {THRESHOLD:.2f}")
print(f"  • Margin Veto Thresh     : {MARGIN_THRESHOLD:.2f} (Safeguard: {USE_MARGIN_SAFEGUARD})")
print(f"  • NMS Policy             : {"ENABLED" if USE_NMS else "DISABLED (Preserves Raw Candidates)"}")
print(f"  • Prompt Template        : {PROMPT_VERSION}")
print(f"  • Active Metrics         : {", ".join(ACTIVE_METRICS)}")
print("=" * 80)


  📋 EXPERIMENT CONFIGURATION (OPTIMAL PRESET)
  • Data Directory         : /Users/hmd/Documents/Workspace/Unified_Test/data/new/400Abstracts/no_disease/balanced/val
  • Label Schema           : NEW (Relabeled) (USE_NEW_LABELS = True)
  • Batch Size (B)         : 5 abstracts per batch
  • Gating Operator        : PROB_OR
  • Weight Combination     : w_novelty=0.30, w_uncertainty=0.70
  • Uncertainty Metric     : MARGIN (MCD passes=5)
  • Novelty Metric         : LOF
  • Threshold (Tau)        : 0.35
  • Margin Veto Thresh     : 0.50 (Safeguard: True)
  • NMS Policy             : DISABLED (Preserves Raw Candidates)
  • Prompt Template        : v1_with_label
  • Active Metrics         : strict, partial, mention


In [4]:
# ==============================================================================
# [CELL 4]: Load Dataset Abstracts and Ground Truth Annotations
# ==============================================================================
all_ann = sorted(glob.glob(os.path.join(DATA_DIR, "*.ann")))
if not all_ann:
    all_ann = sorted(glob.glob(os.path.join(DATA_DIR, "**/*.ann"), recursive=True))

selected_files = []
for ann_path in all_ann:
    gt_ents = parse_ann_file(ann_path)
    txt_path = ann_path.replace(".ann", ".txt")
    if os.path.exists(txt_path):
        selected_files.append((txt_path, ann_path, gt_ents))
        
print(f"✅ Successfully loaded {len(selected_files)} paired abstracts (.txt & .ann) from {DATA_DIR}")


✅ Successfully loaded 80 paired abstracts (.txt & .ann) from /Users/hmd/Documents/Workspace/Unified_Test/data/new/400Abstracts/no_disease/balanced/val


In [5]:
# ==============================================================================
# [CELL 5]: Ground Truth Matching & 10-Outcome Audit Classification Helpers
# ==============================================================================
def normalize_text(text: str) -> str:
    return text.strip().lower()

def match_gt_label(span_text, start_char, end_char, gt_ents):
    p_norm = normalize_text(span_text)
    for g in gt_ents:
        g_norm = normalize_text(g["text"])
        exact_char = (start_char == g["start_char"] and end_char == g["end_char"])
        exact_text = (p_norm == g_norm)
        overlap = not (end_char <= g["start_char"] or start_char >= g["end_char"])
        
        if exact_char or exact_text or (overlap and (p_norm in g_norm or g_norm in p_norm)):
            return g["label"]
    return "O"


In [6]:
# ==============================================================================
# [CELL 6]: Initialize LinkNER Pipeline (SpanNER + LLM Engine)
# ==============================================================================
import importlib
from src.architectures.hybrid_linkner.pipeline import LinkNERPipeline

pipeline = LinkNERPipeline(use_new_labels=USE_NEW_LABELS)
print(f"✅ LinkNERPipeline initialized successfully ({pipeline.num_classes} classes, use_new_labels={pipeline.use_new_labels})")


  🚀 LLM Provider: 9Router / OmniRoute Local Gateway (Forced via LLM_PROVIDER=9router)
  📍 Endpoint: http://localhost:20128/v1 | Model: gemini-cli
✅ LinkNERPipeline initialized successfully (10 classes, use_new_labels=True)


In [7]:
# ==============================================================================
# [CELL 7]: Pipeline Initialization (Zero-Overhead Cached Mode & LOF Scorer)
# ==============================================================================
from src.architectures.hybrid_linkner.novelty_detector import build_lof_novelty_detector
from src.architectures.hybrid_linkner.pipeline import LinkNERPipeline

if USE_CACHED_SPANNER:
    print("⚡ [CACHE MODE] Consuming precomputed SpanNER candidates and LOF novelty scores directly from cache.")
    print("   Skipping LOF vector novelty detector refitting.")
    novelty_detector = None
else:
    print("[LOF Novelty] Initializing and fitting LOF Novelty Detector...")
    novelty_detector = build_lof_novelty_detector(k=10, alpha=0.5)
    print(f"[LOF Novelty] Fitted on {len(novelty_detector.matrix)} training exemplars.")

pipeline = LinkNERPipeline(
    novelty_scorer=novelty_detector,
    w_novelty=W_NOVELTY,
    w_uncertainty=W_UNCERTAINTY,
    uncertainty_method=METHOD,
    uncertainty_threshold=THRESHOLD,
    novelty_threshold=NOVELTY_THRESHOLD,
    margin_threshold=MARGIN_THRESHOLD,
    use_margin_safeguard=USE_MARGIN_SAFEGUARD,
    gating_mode=GATING_MODE,
    prompt_version=PROMPT_VERSION,
    use_nms=USE_NMS,
    mcd_passes=MCD_PASSES,
    use_new_labels=USE_NEW_LABELS,
)

print(f"[Pipeline] Initialized Hybrid LinkNER with Gating={GATING_MODE.upper()}, Tau={THRESHOLD}, NMS={USE_NMS}, Prompt={PROMPT_VERSION}")


[LOF Novelty] Initializing LOF Novelty Detector...
  🚀 LLM Provider: 9Router / OmniRoute Local Gateway (Forced via LLM_PROVIDER=9router)
  📍 Endpoint: http://localhost:20128/v1 | Model: gemini-cli
[LOF Novelty] Fitted on 1095 training exemplars.
[Pipeline] Initialized Hybrid LinkNER with Gating=PROB_OR, Tau=0.35, NMS=False, Prompt=v1_with_label


In [8]:
# ==============================================================================
# [CELL 9]: [Option B - Recommended] Fast Batched Multi-Abstract Pipeline Run
# ==============================================================================
import time
print("=" * 96)
print(f"  Executing Hybrid LinkNER Batch Engine (Method: {METHOD.upper()}, Gating: {GATING_MODE.upper()}, Tau={THRESHOLD})")
print("=" * 96)

texts = [f[0] if isinstance(f, str) else open(f[0], 'r', encoding='utf-8').read() for f in selected_files]
doc_ids = [os.path.basename(f[0]).replace(".txt", "") for f in selected_files]
start_time = time.time()

if USE_CACHED_SPANNER:
    print(f"🚀 Loading precomputed SpanNER candidates from cache: {CACHED_SPANNER_DIR}...")
    split_key = get_cache_split_key(DATA_DIR)
    cached_cands_dict = load_cached_spanner_candidates(
        split_key=split_key,
        cache_dir=CACHED_SPANNER_DIR,
        doc_ids=doc_ids
    )
    # SpanNER baseline predictions directly from cache (0.01s)
    spanner_baseline_preds = []
    for d_id in doc_ids:
        cands = cached_cands_dict.get(d_id, {}).get("candidates", [])
        spanner_baseline_preds.append([
            {
                "span_text": c["span_text"],
                "start_char": c["start_char"],
                "end_char": c["end_char"],
                "label": c["predicted_label"],
                "spanner_label": c["predicted_label"],
                "confidence": f"{c.get('prob', 1.0):.4f}",
                "prob": c.get("prob", 1.0)
            }
            for c in cands if c.get("predicted_label", "O") != "O"
        ])
    print(f"⚡ [CACHE HIT] Loaded {sum(len(p) for p in spanner_baseline_preds)} baseline candidate spans across {len(doc_ids)} abstracts in {time.time() - start_time:.3f}s.")

    # Run Hybrid LinkNER with precomputed candidates
    print("🤖 Running Hybrid LinkNER with Prompt v1 (consuming precomputed candidates)...\n")
    all_doc_preds = pipeline.predict_batch(
        texts=texts,
        batch_size=BATCH_SIZE,
        uncertainty_method=METHOD,
        uncertainty_threshold=THRESHOLD,
        novelty_threshold=NOVELTY_THRESHOLD,
        gating_mode=GATING_MODE,
        w_novelty=W_NOVELTY,
        w_uncertainty=W_UNCERTAINTY,
        mcd_passes=MCD_PASSES,
        use_nms=USE_NMS,
        prompt_version=PROMPT_VERSION,
        margin_threshold=MARGIN_THRESHOLD,
        use_margin_safeguard=USE_MARGIN_SAFEGUARD,
        doc_ids=doc_ids,
        precomputed_candidates=cached_cands_dict,
        verbose=True
    )
else:
    # Run Baseline (SpanNER alone)
    print("⚡ Running SpanNER Baseline...")
    spanner_baseline_preds = pipeline.predict_batch(
        texts=texts,
        batch_size=BATCH_SIZE,
        uncertainty_threshold=1.01,
        use_nms=USE_NMS,
        verbose=False
    )

    # Run Hybrid LinkNER (SpanNER + LOF + Gating + Single-Pass LLM)
    print("🤖 Running Hybrid LinkNER with Prompt v1...")
    all_doc_preds = pipeline.predict_batch(
        texts=texts,
        batch_size=BATCH_SIZE,
        uncertainty_method=METHOD,
        uncertainty_threshold=THRESHOLD,
        novelty_threshold=NOVELTY_THRESHOLD,
        gating_mode=GATING_MODE,
        w_novelty=W_NOVELTY,
        w_uncertainty=W_UNCERTAINTY,
        mcd_passes=MCD_PASSES,
        use_nms=USE_NMS,
        prompt_version=PROMPT_VERSION,
        margin_threshold=MARGIN_THRESHOLD,
        use_margin_safeguard=USE_MARGIN_SAFEGUARD,
        doc_ids=doc_ids,
        verbose=True
    )

elapsed = time.time() - start_time
all_doc_gts = [f[2] for f in selected_files]

total_spans = sum(len(p) for p in all_doc_preds)
escalated_spans = sum(sum(1 for e in p if e.get('escalated_to_llm', False)) for p in all_doc_preds)
local_spans = total_spans - escalated_spans
local_pct = (local_spans / total_spans * 100.0) if total_spans > 0 else 100.0

print(f"\n✅ Completed in {elapsed:.2f}s ({elapsed/max(1, len(texts)):.2f}s / abstract).")
print(f"📊 Local Computation % Saved: {local_pct:.2f}% (Locally Accepted: {local_spans}/{total_spans}, Escalated: {escalated_spans})")


  Executing Hybrid LinkNER Batch Engine (Method: MARGIN, Gating: PROB_OR, Tau=0.35)
⚡ Running SpanNER Baseline...
🤖 Running Hybrid LinkNER with Prompt v1...
  🚀 LinkNER Batch Engine: Processing 80 abstracts in 16 batches (B=5)
  ⚡ Method: MARGIN | Gating Strategy: LOF Novelty Detector [PROB_OR Union Gating: 1-(1-N)*(1-U) >= 0.35]
  🎯 NMS Policy: DISABLED (Raw Candidates Preserved) | Prompt: v1_with_label
  🏷️  Schema: New Relabeled Ontology (use_new_labels=True)

📦 [Batch 1/16] Processing abstracts 1-5 of 80 (Size: 5)
  ⚡ [Stage 1: SpanNER] Forward pass evaluated 14565 span predictions across 5 abstracts.
  🔍 [Stage 2: LOF Novelty] Embedded and scored 97 non-background mentions in parallel.
  🤖 [Stage 3: LLM Escalation] Escalating 86 unreliable spans across 5 abstracts concurrently (v1_with_label)...
     -> Abstract 1: Escalated 18 spans -> 18 arbitrated decisions (15 positive kept, 3 pruned to 'O').
     -> Abstract 2: Escalated 17 spans -> 17 arbitrated decisions (17 positive kept, 

## 🏆 Standardized Multi-Tier Evaluation Framework: Dual-View Benchmark

This evaluation framework decouples the performance of the **two architectural stages** for the Single-Call Hybrid LinkNER pipeline:

### 1. View 1: Candidate-Level Arbitration Benchmark (Master Report / Sweep Engine Formulation)
- **Evaluates**: The decision accuracy of the Single-Call LLM reasoning engine on candidate spans proposed by SpanNER and audited by Gating.
- **Recall Denominator**: Evaluated candidate spans matching ground truth (`TP / (TP + FN)` across candidate mentions).
- **Baseline Comparison**: Evaluates how much the LLM improves or degrades SpanNER's proposed candidate pool.

### 2. View 2: Global End-to-End Dataset Benchmark (Standard Academic Gold Set Recovery)
- **Evaluates**: Total ground truth entity recovery against all **1,174 gold entities** in the dataset across standard academic formulations:
  - **Strict CoNLL (1)**: Exact character boundary + exact category match.
  - **Partial MUC-7 (3)**: Any character boundary overlap + exact category match (1-to-1 bipartite lockout).
  - **Candidate Mention (4)**: Candidate mention coverage without 1-to-1 lockout against all 1,174 gold entities.
- **Unseen Span Discovery Impact**: Evaluates whether allowing the LLM to discover missed spans (`v1_allowed_new_span`) increases global recall beyond SpanNER's initial candidate proposal ceiling.

In [9]:
# ==============================================================================
# [CELL 10]: Standardized Multi-Tier Benchmark (Dual-View Evaluation)
# View 1: Candidate Arbitration Benchmark (Sweep Engine / Master Report Formulation)
# View 2: Global End-to-End Dataset Benchmark (Standard Academic Gold Set Recovery)
# View 3: Grounded 10-Outcome Diagnostic Breakdown (Canonical Publication Taxonomy)
# ==============================================================================
import numpy as np
from collections import Counter
from src.common.evaluation_framework import (
    evaluate_doc_4_formulations,
    aggregate_4_formulations,
    format_4_formulations_summary_box
)
from src.common.audit_logger import (
    classify_10_outcome_str,
    match_gt_label,
    CANONICAL_OUTCOME_CATEGORIES,
    compute_canonical_audit_breakdown,
    format_canonical_audit_table
)
from src.common.label_mapping import canonicalize_label

# ------------------------------------------------------------------------------
# Build Flat Audit Records across all abstracts
# ------------------------------------------------------------------------------
all_audit_records = []
all_outcomes = []
escalated_outcomes = []
unescalated_outcomes = []
newly_discovered_records = []

for doc_idx, (txt_path, ann_path, gt_ents) in enumerate(selected_files):
    preds = all_doc_preds[doc_idx]
    for p in preds:
        gt_lbl = match_gt_label(p["span_text"], p["start_char"], p["end_char"], gt_ents)
        s_lbl = canonicalize_label(p.get("spanner_label", p.get("label", "O")), use_new_labels=USE_NEW_LABELS)
        f_lbl = canonicalize_label(p.get("label", "O"), use_new_labels=USE_NEW_LABELS)
        is_esc = p.get("escalated_to_llm", False)
        is_new = "Unseen Span Discovery" in p.get("source", "")
        
        rec = {
            "span": p["span_text"],
            "start_char": p["start_char"],
            "end_char": p["end_char"],
            "spanner_label": s_lbl,
            "hybrid_label": f_lbl,
            "ground_truth": gt_lbl,
            "escalated_to_llm": is_esc,
            "is_new_discovered": is_new,
            "source": p.get("source", "")
        }
        all_audit_records.append(rec)
        if is_new:
            newly_discovered_records.append(rec)
            
        outcome_str = classify_10_outcome_str(gt_label=gt_lbl, spanner_label=s_lbl, final_label=f_lbl, escalated=is_esc)
        all_outcomes.append(outcome_str)
        if is_esc:
            escalated_outcomes.append(outcome_str)
        else:
            unescalated_outcomes.append(outcome_str)

# ------------------------------------------------------------------------------
# 1. VIEW 1: Candidate-Level Arbitration Benchmark (Master Report Formulation)
# ------------------------------------------------------------------------------
classes = sorted(list(set(r["ground_truth"] for r in all_audit_records if r["ground_truth"] != "O") |
                      set(r["hybrid_label"] for r in all_audit_records if r["hybrid_label"] != "O") |
                      set(r["spanner_label"] for r in all_audit_records if r["spanner_label"] != "O")))

per_class_table = []
tot_s_tp = tot_s_fp = tot_s_fn = 0
tot_h_tp = tot_h_fp = tot_h_fn = 0
s_f1_list, h_f1_list = [], []

for cls in classes:
    s_tp = sum(1 for r in all_audit_records if r["spanner_label"] == cls and r["ground_truth"] == cls)
    s_fp = sum(1 for r in all_audit_records if r["spanner_label"] == cls and r["ground_truth"] != cls)
    s_fn = sum(1 for r in all_audit_records if r["ground_truth"] == cls and r["spanner_label"] != cls)
    
    h_tp = sum(1 for r in all_audit_records if r["hybrid_label"] == cls and r["ground_truth"] == cls)
    h_fp = sum(1 for r in all_audit_records if r["hybrid_label"] == cls and r["ground_truth"] != cls)
    h_fn = sum(1 for r in all_audit_records if r["ground_truth"] == cls and r["hybrid_label"] != cls)
    
    s_p = s_tp / (s_tp + s_fp) if (s_tp + s_fp) > 0 else 0.0
    s_r = s_tp / (s_tp + s_fn) if (s_tp + s_fn) > 0 else 0.0
    s_f1 = (2 * s_p * s_r) / (s_p + s_r) if (s_p + s_r) > 0 else 0.0
    
    h_p = h_tp / (h_tp + h_fp) if (h_tp + h_fp) > 0 else 0.0
    h_r = h_tp / (h_tp + h_fn) if (h_tp + h_fn) > 0 else 0.0
    h_f1 = (2 * h_p * h_r) / (h_p + h_r) if (h_p + h_r) > 0 else 0.0
    
    tot_s_tp += s_tp; tot_s_fp += s_fp; tot_s_fn += s_fn
    tot_h_tp += h_tp; tot_h_fp += h_fp; tot_h_fn += h_fn
    
    s_f1_list.append(s_f1); h_f1_list.append(h_f1)
    delta = h_f1 - s_f1
    delta_str = f"+{delta*100:.2f}%" if delta >= 0 else f"{delta*100:.2f}%"
    per_class_table.append((cls, s_f1, h_f1, delta_str, h_tp, h_fp, h_fn))

s_mic_p = tot_s_tp / (tot_s_tp + tot_s_fp) if (tot_s_tp + tot_s_fp) > 0 else 0.0
s_mic_r = tot_s_tp / (tot_s_tp + tot_s_fn) if (tot_s_tp + tot_s_fn) > 0 else 0.0
s_mic_f1 = (2 * s_mic_p * s_mic_r) / (s_mic_p + s_mic_r) if (s_mic_p + s_mic_r) > 0 else 0.0
s_mac_f1 = float(np.mean(s_f1_list)) if s_f1_list else 0.0

h_mic_p = tot_h_tp / (tot_h_tp + tot_h_fp) if (tot_h_tp + tot_h_fp) > 0 else 0.0
h_mic_r = tot_h_tp / (tot_h_tp + tot_h_fn) if (tot_h_tp + tot_h_fn) > 0 else 0.0
h_mic_f1 = (2 * h_mic_p * h_mic_r) / (h_mic_p + h_mic_r) if (h_mic_p + h_mic_r) > 0 else 0.0
h_mac_f1 = float(np.mean(h_f1_list)) if h_f1_list else 0.0

print("=" * 114)
print("🏆 VIEW 1: CANDIDATE-LEVEL ARBITRATION BENCHMARK (Sweep Engine / Master Report Formulation)")
print(f"   Architecture: Single-Call Hybrid LinkNER | Prompt: {PROMPT_VERSION}")
print("=" * 114)
print(f"{'Class Name':<48} | {'SpanNER F1':<10} | {'Hybrid F1':<10} | {'Delta':<8} | {'TP':<5} | {'FP':<5} | {'FN':<5}")
print("-" * 114)
for cls, s_f1, h_f1, d_str, tp, fp, fn in per_class_table:
    print(f"{cls:<48} | {s_f1*100:8.2f}% | {h_f1*100:8.2f}% | {d_str:<8} | {tp:<5} | {fp:<5} | {fn:<5}")
print("-" * 114)
print(f"Candidate Micro Precision : {h_mic_p*100:.2f}%  (SpanNER: {s_mic_p*100:.2f}%, Gain: +{(h_mic_p-s_mic_p)*100:.2f}%)")
print(f"Candidate Micro Recall    : {h_mic_r*100:.2f}%  (SpanNER: {s_mic_r*100:.2f}%, Gain: +{(h_mic_r-s_mic_r)*100:.2f}%)")
print(f"Candidate Micro F1-Score  : {h_mic_f1*100:.2f}%  (SpanNER: {s_mic_f1*100:.2f}%, Gain: +{(h_mic_f1-s_mic_f1)*100:.2f}%)")
print(f"Macro F1-Score (Per-Class): {h_mac_f1*100:.2f}%  (SpanNER: {s_mac_f1*100:.2f}%, Gain: +{(h_mac_f1-s_mac_f1)*100:.2f}%)\n")

# ------------------------------------------------------------------------------
# 2. VIEW 2: Global End-to-End Dataset Benchmark (Standard Academic Formulation)
# ------------------------------------------------------------------------------
all_doc_gts = [f[2] for f in selected_files]
spanner_doc_m = [evaluate_doc_4_formulations(spanner_baseline_preds[i], all_doc_gts[i], use_new_labels=USE_NEW_LABELS) for i in range(len(selected_files))]
hybrid_doc_m = [evaluate_doc_4_formulations(all_doc_preds[i], all_doc_gts[i], use_new_labels=USE_NEW_LABELS) for i in range(len(selected_files))]

spanner_results = aggregate_4_formulations(spanner_doc_m)
hybrid_results = aggregate_4_formulations(hybrid_doc_m)

system_name = f"Single-Call Hybrid LinkNER ({PROMPT_VERSION})"
summary_box = format_4_formulations_summary_box(
    spanner_results=spanner_results,
    hybrid_results=hybrid_results,
    active_metrics=["strict", "partial", "mention"],
    system_name=system_name
)
print(summary_box)

total_gold_entities = sum(len(gts) for gts in all_doc_gts)
covered_gold_entities = hybrid_results["mention"]["tp"]
missed_proposals = total_gold_entities - covered_gold_entities

print("\n" + "=" * 114)
print("💡 ARCHITECTURAL PERFORMANCE ANALYSIS:")
print(f"• Candidate-Level Arbitration F1 : {h_mic_f1*100:.2f}% Micro F1 on evaluated candidates.")
print(f"• Global End-to-End Recovery     : Evaluated across all {total_gold_entities} gold entities in the dataset.")
print(f"  - Strict CoNLL (1)   : {hybrid_results['strict']['f1']*100:.2f}% F1 (P: {hybrid_results['strict']['p']*100:.2f}%, R: {hybrid_results['strict']['r']*100:.2f}%)")
print(f"  - Partial MUC-7 (3)  : {hybrid_results['partial']['f1']*100:.2f}% F1 (P: {hybrid_results['partial']['p']*100:.2f}%, R: {hybrid_results['partial']['r']*100:.2f}%)")
print(f"  - Candidate Mention  : {hybrid_results['mention']['f1']*100:.2f}% F1 (P: {hybrid_results['mention']['p']*100:.2f}%, R: {hybrid_results['mention']['r']*100:.2f}%)")
if newly_discovered_records:
    print(f"• Unseen Span Discovery          : {len(newly_discovered_records)} new entities discovered and classified by LLM!")
    for nd in newly_discovered_records[:5]:
        print(f"    - \"{nd['span']}\" -> {nd['hybrid_label']} (Ground Truth: {nd['ground_truth']})")
    if len(newly_discovered_records) > 5:
        print(f"    ... and {len(newly_discovered_records) - 5} more.")
print("=" * 114)

# ------------------------------------------------------------------------------
# 3. Grounded 10-Outcome Diagnostic Breakdown Matching Canonical Publication Taxonomy
# ------------------------------------------------------------------------------
print("\n" + format_canonical_audit_table(
    breakdown_or_preds=all_doc_preds,
    all_doc_gts=all_doc_gts,
    method=METHOD,
    use_new_labels=USE_NEW_LABELS
))


🏆 VIEW 1: CANDIDATE-LEVEL ARBITRATION BENCHMARK (Sweep Engine / Master Report Formulation)
   Architecture: Single-Call Hybrid LinkNER | Prompt: v1_with_label
Class Name                                       | SpanNER F1 | Hybrid F1  | Delta    | TP    | FP    | FN   
------------------------------------------------------------------------------------------------------------------
Environmental_exposures                          |    75.44% |    80.68% | +5.23%   | 167   | 46    | 34   
Mental_health_practices                          |    84.89% |    91.03% | +6.14%   | 66    | 3     | 10   
Non_physical_leisure_time_activities             |    63.74% |    83.76% | +20.02%  | 49    | 11    | 8    
Nutrition                                        |    77.93% |    85.17% | +7.24%   | 313   | 82    | 27   
Personal_care_products_and_cosmetic_procedures   |    73.21% |    85.94% | +12.72%  | 55    | 5     | 13   
Physical_activities                              |    78.92% |    89.07% | +

In [10]:
# ==============================================================================
# [CELL 11]: Pipeline Stage Progression & 3-Tier Multi-Metric Error Analysis
# ==============================================================================
import os
import torch
from collections import defaultdict
from transformers import RobertaForTokenClassification, AutoTokenizer
from src.common.label_mapping import canonicalize_label
from src.common.evaluation_framework import evaluate_doc_4_formulations, aggregate_4_formulations

print("=" * 105)
print(f"  🔍 Pipeline Stage Error Analysis | Governed by Global: USE_NEW_LABELS = {USE_NEW_LABELS} ({schema_name} Ontology)")
print("=" * 105)

def evaluate_stage_predictions(all_stage_preds, all_doc_gts, use_new_labels=True):
    doc_metrics = [evaluate_doc_4_formulations(preds, gts, use_new_labels=use_new_labels) for preds, gts in zip(all_stage_preds, all_doc_gts)]
    agg = aggregate_4_formulations(doc_metrics)
    
    abs_stats = {
        m: {
            "p": [d[m]["p"] for d in doc_metrics],
            "r": [d[m]["r"] for d in doc_metrics],
            "f1": [d[m]["f1"] for d in doc_metrics]
        } for m in ["strict", "partial", "mention"]
    }
    
    class_counts = defaultdict(lambda: {"strict": {"tp": 0, "pred": 0, "gt": 0}, "partial": {"tp": 0, "pred": 0, "gt": 0}, "mention": {"tp": 0, "pred": 0, "gt": 0}})
    
    for preds, gts in zip(all_stage_preds, all_doc_gts):
        active_gts = [g for g in gts if canonicalize_label(g.get("label", "O"), use_new_labels=use_new_labels) not in {"O", "Lifestyle_factor", "LSF_out_of_context", "Non_LSF", "Non-LSF"}]
        active_preds = [p for p in preds if canonicalize_label(p.get("label", p.get("spanner_label", "O")), use_new_labels=use_new_labels) not in {"O", "Lifestyle_factor", "LSF_out_of_context", "Non_LSF", "Non-LSF"}]
        
        for g in active_gts:
            lbl = canonicalize_label(g.get("label", "O"), use_new_labels=use_new_labels)
            class_counts[lbl]["strict"]["gt"] += 1
            class_counts[lbl]["partial"]["gt"] += 1
            class_counts[lbl]["mention"]["gt"] += 1
            
        for p in active_preds:
            lbl = canonicalize_label(p.get("label", p.get("spanner_label", "O")), use_new_labels=use_new_labels)
            class_counts[lbl]["strict"]["pred"] += 1
            class_counts[lbl]["partial"]["pred"] += 1
            class_counts[lbl]["mention"]["pred"] += 1

        # Strict matches
        matched_gt_s = set()
        for p in active_preds:
            p_s, p_e = p.get("start_char", -1), p.get("end_char", -1)
            p_lbl = canonicalize_label(p.get("label", p.get("spanner_label", "O")), use_new_labels=use_new_labels)
            for g_idx, g in enumerate(active_gts):
                if g_idx in matched_gt_s: continue
                g_s, g_e = g.get("start_char", -1), g.get("end_char", -1)
                g_lbl = canonicalize_label(g.get("label", "O"), use_new_labels=use_new_labels)
                if p_s == g_s and p_e == g_e and p_lbl.lower() == g_lbl.lower():
                    matched_gt_s.add(g_idx)
                    class_counts[g_lbl]["strict"]["tp"] += 1
                    break
                    
        # Partial matches
        matched_gt_p = set()
        for p in active_preds:
            p_s, p_e = p.get("start_char", -1), p.get("end_char", -1)
            p_lbl = canonicalize_label(p.get("label", p.get("spanner_label", "O")), use_new_labels=use_new_labels)
            for g_idx, g in enumerate(active_gts):
                if g_idx in matched_gt_p: continue
                g_s, g_e = g.get("start_char", -1), g.get("end_char", -1)
                g_lbl = canonicalize_label(g.get("label", "O"), use_new_labels=use_new_labels)
                overlap = not (p_e <= g_s or p_s >= g_e)
                if overlap and p_lbl.lower() == g_lbl.lower():
                    matched_gt_p.add(g_idx)
                    class_counts[g_lbl]["partial"]["tp"] += 1
                    break

        # Mention matches
        for p in active_preds:
            p_s, p_e = p.get("start_char", -1), p.get("end_char", -1)
            p_lbl = canonicalize_label(p.get("label", p.get("spanner_label", "O")), use_new_labels=use_new_labels)
            for g in active_gts:
                g_s, g_e = g.get("start_char", -1), g.get("end_char", -1)
                g_lbl = canonicalize_label(g.get("label", "O"), use_new_labels=use_new_labels)
                if not (p_e <= g_s or p_s >= g_e):
                    if p_lbl.lower() == g_lbl.lower():
                        class_counts[g_lbl]["mention"]["tp"] += 1
                    break

    res = {}
    for m in ["strict", "partial", "mention"]:
        m_data = agg[m]
        tp, fp, fn = m_data["tp"], m_data["fp"], m_data["fn"]
        micro_p = m_data["p"]
        micro_r = m_data["r"]
        micro_f1 = m_data["f1"]
        
        macro_abs_p = sum(abs_stats[m]["p"]) / max(1, len(abs_stats[m]["p"]))
        macro_abs_r = sum(abs_stats[m]["r"]) / max(1, len(abs_stats[m]["r"]))
        macro_abs_f1 = sum(abs_stats[m]["f1"]) / max(1, len(abs_stats[m]["f1"]))
        
        cls_f1s, cls_ps, cls_rs = [], [], []
        for c, data in class_counts.items():
            ct = data[m]
            cp = ct["tp"] / ct["pred"] if ct["pred"] > 0 else 0.0
            cr = ct["tp"] / ct["gt"] if ct["gt"] > 0 else 0.0
            cf1 = (2 * cp * cr) / (cp + cr) if (cp + cr) > 0 else 0.0
            cls_ps.append(cp); cls_rs.append(cr); cls_f1s.append(cf1)
            
        macro_cls_p = sum(cls_ps) / max(1, len(cls_ps))
        macro_cls_r = sum(cls_rs) / max(1, len(cls_rs))
        macro_cls_f1 = sum(cls_f1s) / max(1, len(cls_f1s))
        
        res[m] = {
            "total_gt": tp + fn,
            "total_pred": tp + fp,
            "tp": tp, "fp": fp, "fn": fn,
            "micro_p": micro_p, "micro_r": micro_r, "micro_f1": micro_f1,
            "macro_abs_p": macro_abs_p, "macro_abs_r": macro_abs_r, "macro_abs_f1": macro_abs_f1,
            "macro_cls_p": macro_cls_p, "macro_cls_r": macro_cls_r, "macro_cls_f1": macro_cls_f1
        }
    return res

def print_stage_table(title, res):
    s = res["strict"]
    p = res["partial"]
    m = res["mention"]
    
    s_tp_str = f"{s['tp']} / {s['total_gt']}"
    p_tp_str = f"{p['tp']} / {p['total_gt']}"
    m_tp_str = f"{m['tp']} / {m['total_gt']}"
    
    s_mp_str = f"{s['micro_p']*100:.2f}% ({s['micro_p']:.4f})"
    p_mp_str = f"{p['micro_p']*100:.2f}% ({p['micro_p']:.4f})"
    m_mp_str = f"{m['micro_p']*100:.2f}% ({m['micro_p']:.4f})"
    
    s_mr_str = f"{s['micro_r']*100:.2f}% ({s['micro_r']:.4f})"
    p_mr_str = f"{p['micro_r']*100:.2f}% ({p['micro_r']:.4f})"
    m_mr_str = f"{m['micro_r']*100:.2f}% ({m['micro_r']:.4f})"
    
    s_mf_str = f"{s['micro_f1']*100:.2f}% ({s['micro_f1']:.4f})"
    p_mf_str = f"{p['micro_f1']*100:.2f}% ({p['micro_f1']:.4f})"
    m_mf_str = f"{m['micro_f1']*100:.2f}% ({m['micro_f1']:.4f})"
    
    s_af_str = f"{s['macro_abs_f1']*100:.2f}% ({s['macro_abs_f1']:.4f})"
    p_af_str = f"{p['macro_abs_f1']*100:.2f}% ({p['macro_abs_f1']:.4f})"
    m_af_str = f"{m['macro_abs_f1']*100:.2f}% ({m['macro_abs_f1']:.4f})"
    
    s_cf_str = f"{s['macro_cls_f1']*100:.2f}% ({s['macro_cls_f1']:.4f})"
    p_cf_str = f"{p['macro_cls_f1']*100:.2f}% ({p['macro_cls_f1']:.4f})"
    m_cf_str = f"{m['macro_cls_f1']*100:.2f}% ({m['macro_cls_f1']:.4f})"
    
    print("=" * 105)
    print(f"  {title.upper()}")
    print("=" * 105)
    print(f"{'Metric':<32} | {'Strict (Exact CoNLL)':<22} | {'Partial (MUC-7)':<22} | {'Candidate Mention':<22}")
    print("-" * 105)
    print(f"{'Total Ground Truth Entities':<32} | {s['total_gt']:<22} | {p['total_gt']:<22} | {m['total_gt']:<22}")
    print(f"{'Total Predicted Entities':<32} | {s['total_pred']:<22} | {p['total_pred']:<22} | {m['total_pred']:<22}")
    print(f"{'True Positives (TP)':<32} | {s_tp_str:<22} | {p_tp_str:<22} | {m_tp_str:<22}")
    print(f"{'False Positives (FP)':<32} | {s['fp']:<22} | {p['fp']:<22} | {m['fp']:<22}")
    print(f"{'False Negatives (FN)':<32} | {s['fn']:<22} | {p['fn']:<22} | {m['fn']:<22}")
    print("-" * 105)
    print(f"{'Micro Precision':<32} | {s_mp_str:<22} | {p_mp_str:<22} | {m_mp_str:<22}")
    print(f"{'Micro Recall':<32} | {s_mr_str:<22} | {p_mr_str:<22} | {m_mr_str:<22}")
    print(f"{'Micro F1-Score':<32} | {s_mf_str:<22} | {p_mf_str:<22} | {m_mf_str:<22}")
    print("-" * 105)
    print(f"{'Macro F1 (Per-Abstract)':<32} | {s_af_str:<22} | {p_af_str:<22} | {m_af_str:<22}")
    print(f"{'Macro F1 (Per-Class)':<32} | {s_cf_str:<22} | {p_cf_str:<22} | {m_cf_str:<22}")
    print("=" * 105 + "\n")

# ------------------------------------------------------------------------------
# 2. Extract Step 1: RoBERTa Token Classification Predictions
# ------------------------------------------------------------------------------
BIO21 = [
    "O", "I-Socioeconomic_factors", "I-Sleep", "I-Physical_activity", "I-Nutrition",
    "I-Non_physical_leisure_time_activities", "I-Mental_health_practices", "I-Lifestyle_factor",
    "I-Environmental_exposures", "I-Drugs", "I-Beauty_and_Cleaning",
    "B-Socioeconomic_factors", "B-Sleep", "B-Physical_activity", "B-Nutrition",
    "B-Non_physical_leisure_time_activities", "B-Mental_health_practices", "B-Lifestyle_factor",
    "B-Environmental_exposures", "B-Drugs", "B-Beauty_and_Cleaning",
]
BIO21_ID2LABEL = {i: l for i, l in enumerate(BIO21)}

def bio_tokens_to_entities(text, bio_tags, offsets):
    ents = []
    cur_start, cur_end, cur_label = None, None, None
    def flush():
        nonlocal cur_start, cur_end, cur_label
        if cur_start is not None and cur_label:
            span_text = text[cur_start:cur_end].strip()
            if span_text:
                ents.append({"span_text": span_text, "start_char": cur_start, "end_char": cur_end, "label": cur_label})
        cur_start, cur_end, cur_label = None, None, None

    for tag, (cs, ce) in zip(bio_tags, offsets):
        if cs == ce or tag == "O":
            flush()
            continue
        lab = tag[2:] if (tag.startswith("B-") or tag.startswith("I-")) else tag
        if tag.startswith("B-"):
            flush()
            cur_start, cur_end, cur_label = cs, ce, lab
        elif tag.startswith("I-"):
            if cur_label == lab and cur_end == cs:
                cur_end = ce
            else:
                flush()
                cur_start, cur_end, cur_label = cs, ce, lab
    flush()
    return ents

roberta_doc_preds = []
try:
    TOKEN_MODEL_DIR = os.path.join(PROJECT_ROOT, "models/NER_Model/trained_NER_model")
    if os.path.exists(TOKEN_MODEL_DIR) and "selected_files" in globals():
        print("⚡ [Step 1] Loading RoBERTa token classification baseline model...")
        device = "mps" if torch.backends.mps.is_available() else ("cuda" if torch.cuda.is_available() else "cpu")
        tok_model = RobertaForTokenClassification.from_pretrained(TOKEN_MODEL_DIR).to(device).eval()
        tokenizer = AutoTokenizer.from_pretrained(TOKEN_MODEL_DIR)
        
        with torch.no_grad():
            for txt_path, ann_path, gt_ents in selected_files:
                with open(txt_path, "r", encoding="utf-8") as f:
                    text = f.read()
                enc = tokenizer(text, return_offsets_mapping=True, return_tensors="pt", max_length=512, truncation=True)
                ids = enc["input_ids"].to(device)
                mask = enc["attention_mask"].to(device)
                offsets = enc["offset_mapping"][0].tolist()
                logits = tok_model(input_ids=ids, attention_mask=mask).logits[0]
                pred_ids = logits.argmax(dim=-1).tolist()
                tags = [BIO21_ID2LABEL[i] for i in pred_ids]
                doc_ents = bio_tokens_to_entities(text, tags, offsets)
                roberta_doc_preds.append(doc_ents)
        print(f"✅ [Step 1] RoBERTa token baseline evaluated on {len(roberta_doc_preds)} abstracts.")
    else:
        print("⚠️ [Step 1] RoBERTa checkpoint directory or selected_files not found. Skipping Step 1.")
except Exception as e:
    print(f"⚠️ [Step 1] Skipped RoBERTa token model evaluation: {e}")

# ------------------------------------------------------------------------------
# 3. Extract Step 2: Pure SpanNER Predictions (Before LLM Escalation)
# ------------------------------------------------------------------------------
spanner_doc_preds = spanner_baseline_preds

# ------------------------------------------------------------------------------
# 4. Extract Step 3: Full Pipeline Predictions (SpanNER + LLM Arbitrated)
# ------------------------------------------------------------------------------
full_pipeline_doc_preds = all_doc_preds

# ------------------------------------------------------------------------------
# 5. Evaluate and Print Individual Stage Tables
# ------------------------------------------------------------------------------
results_step1 = evaluate_stage_predictions(roberta_doc_preds, all_doc_gts, use_new_labels=USE_NEW_LABELS) if roberta_doc_preds else None
results_step2 = evaluate_stage_predictions(spanner_doc_preds, all_doc_gts, use_new_labels=USE_NEW_LABELS)
results_step3 = evaluate_stage_predictions(full_pipeline_doc_preds, all_doc_gts, use_new_labels=USE_NEW_LABELS)

if results_step1:
    print_stage_table("1 - RoBERTa Token Classification vs Ground Truth", results_step1)

print_stage_table("2 - SpanNER Neural Span Baseline vs Ground Truth", results_step2)
print_stage_table("3 - SpanNER + LLM Calls (Full LinkNER Pipeline) vs Ground Truth", results_step3)

# ------------------------------------------------------------------------------
# 6. Side-by-Side Pipeline Progression & Error Analysis Table (All 3 Metrics)
# ------------------------------------------------------------------------------
print("=" * 115)
print("                           PIPELINE PROGRESSION & ERROR ANALYSIS (3-STEP COMPARISON)")
print("=" * 115)
hdr_step1 = "Step 1: RoBERTa Baseline" if results_step1 else "Step 1: RoBERTa (N/A)"
print(f"{'Metric':<34} | {hdr_step1:<24} | {'Step 2: SpanNER Baseline':<24} | {'Step 3: SpanNER + LLM (Full)':<26}")
print("-" * 115)

def fmt_val(r, mode, key, is_pct=False):
    if not r or mode not in r or key not in r[mode]:
        return "N/A"
    val = r[mode][key]
    if is_pct:
        return f"{val * 100:.2f}%"
    return str(val)

def fmt_counts(r, mode):
    if not r or mode not in r:
        return "N/A"
    tp = r[mode]["tp"]
    fp = r[mode]["fp"]
    fn = r[mode]["fn"]
    return f"{tp}/{fp}/{fn}"

# Predicted & Counts
print(f"{'Total Predicted Entities':<34} | {fmt_val(results_step1, 'strict', 'total_pred'):<24} | {fmt_val(results_step2, 'strict', 'total_pred'):<24} | {fmt_val(results_step3, 'strict', 'total_pred'):<26}")
print(f"{'Strict TP / FP / FN':<34} | {fmt_counts(results_step1, 'strict'):<24} | {fmt_counts(results_step2, 'strict'):<24} | {fmt_counts(results_step3, 'strict'):<26}")
print(f"{'Partial TP / FP / FN':<34} | {fmt_counts(results_step1, 'partial'):<24} | {fmt_counts(results_step2, 'partial'):<24} | {fmt_counts(results_step3, 'partial'):<26}")
print(f"{'Candidate Mention TP / FP / FN':<34} | {fmt_counts(results_step1, 'mention'):<24} | {fmt_counts(results_step2, 'mention'):<24} | {fmt_counts(results_step3, 'mention'):<26}")
print("-" * 115)

# Strict Metrics
print(f"{'Strict Micro Precision':<34} | {fmt_val(results_step1, 'strict', 'micro_p', True):<24} | {fmt_val(results_step2, 'strict', 'micro_p', True):<24} | {fmt_val(results_step3, 'strict', 'micro_p', True):<26}")
print(f"{'Strict Micro Recall':<34} | {fmt_val(results_step1, 'strict', 'micro_r', True):<24} | {fmt_val(results_step2, 'strict', 'micro_r', True):<24} | {fmt_val(results_step3, 'strict', 'micro_r', True):<26}")
print(f"{'Strict Micro F1-Score':<34} | {fmt_val(results_step1, 'strict', 'micro_f1', True):<24} | {fmt_val(results_step2, 'strict', 'micro_f1', True):<24} | {fmt_val(results_step3, 'strict', 'micro_f1', True):<26}")
print(f"{'Strict Macro F1 (Per-Abstract)':<34} | {fmt_val(results_step1, 'strict', 'macro_abs_f1', True):<24} | {fmt_val(results_step2, 'strict', 'macro_abs_f1', True):<24} | {fmt_val(results_step3, 'strict', 'macro_abs_f1', True):<26}")
print(f"{'Strict Macro F1 (Per-Class)':<34} | {fmt_val(results_step1, 'strict', 'macro_cls_f1', True):<24} | {fmt_val(results_step2, 'strict', 'macro_cls_f1', True):<24} | {fmt_val(results_step3, 'strict', 'macro_cls_f1', True):<26}")
print("-" * 115)

# Partial Metrics
print(f"{'Partial Micro Precision':<34} | {fmt_val(results_step1, 'partial', 'micro_p', True):<24} | {fmt_val(results_step2, 'partial', 'micro_p', True):<24} | {fmt_val(results_step3, 'partial', 'micro_p', True):<26}")
print(f"{'Partial Micro Recall':<34} | {fmt_val(results_step1, 'partial', 'micro_r', True):<24} | {fmt_val(results_step2, 'partial', 'micro_r', True):<24} | {fmt_val(results_step3, 'partial', 'micro_r', True):<26}")
print(f"{'Partial Micro F1-Score':<34} | {fmt_val(results_step1, 'partial', 'micro_f1', True):<24} | {fmt_val(results_step2, 'partial', 'micro_f1', True):<24} | {fmt_val(results_step3, 'partial', 'micro_f1', True):<26}")
print(f"{'Partial Macro F1 (Per-Abstract)':<34} | {fmt_val(results_step1, 'partial', 'macro_abs_f1', True):<24} | {fmt_val(results_step2, 'partial', 'macro_abs_f1', True):<24} | {fmt_val(results_step3, 'partial', 'macro_abs_f1', True):<26}")
print(f"{'Partial Macro F1 (Per-Class)':<34} | {fmt_val(results_step1, 'partial', 'macro_cls_f1', True):<24} | {fmt_val(results_step2, 'partial', 'macro_cls_f1', True):<24} | {fmt_val(results_step3, 'partial', 'macro_cls_f1', True):<26}")
print("-" * 115)

# Candidate Mention Metrics
print(f"{'Candidate Mention Micro Precision':<34} | {fmt_val(results_step1, 'mention', 'micro_p', True):<24} | {fmt_val(results_step2, 'mention', 'micro_p', True):<24} | {fmt_val(results_step3, 'mention', 'micro_p', True):<26}")
print(f"{'Candidate Mention Micro Recall':<34} | {fmt_val(results_step1, 'mention', 'micro_r', True):<24} | {fmt_val(results_step2, 'mention', 'micro_r', True):<24} | {fmt_val(results_step3, 'mention', 'micro_r', True):<26}")
print(f"{'Candidate Mention Micro F1':<34} | {fmt_val(results_step1, 'mention', 'micro_f1', True):<24} | {fmt_val(results_step2, 'mention', 'micro_f1', True):<24} | {fmt_val(results_step3, 'mention', 'micro_f1', True):<26}")
print(f"{'Candidate Mention Macro F1 (Abs)':<34} | {fmt_val(results_step1, 'mention', 'macro_abs_f1', True):<24} | {fmt_val(results_step2, 'mention', 'macro_abs_f1', True):<24} | {fmt_val(results_step3, 'mention', 'macro_abs_f1', True):<26}")
print(f"{'Candidate Mention Macro F1 (Cls)':<34} | {fmt_val(results_step1, 'mention', 'macro_cls_f1', True):<24} | {fmt_val(results_step2, 'mention', 'macro_cls_f1', True):<24} | {fmt_val(results_step3, 'mention', 'macro_cls_f1', True):<26}")
print("=" * 115 + "\n")


  🔍 Pipeline Stage Error Analysis | Governed by Global: USE_NEW_LABELS = True (NEW (Relabeled) Ontology)
⚡ [Step 1] Loading RoBERTa token classification baseline model...
✅ [Step 1] RoBERTa token baseline evaluated on 80 abstracts.
  1 - ROBERTA TOKEN CLASSIFICATION VS GROUND TRUTH
Metric                           | Strict (Exact CoNLL)   | Partial (MUC-7)        | Candidate Mention     
---------------------------------------------------------------------------------------------------------
Total Ground Truth Entities      | 1174                   | 1174                   | 1527                  
Total Predicted Entities         | 2085                   | 2085                   | 2085                  
True Positives (TP)              | 335 / 1174             | 686 / 1174             | 1035 / 1527           
False Positives (FP)             | 1750                   | 1399                   | 1050                  
False Negatives (FN)             | 839                    | 488        

In [11]:
# ==============================================================================
# [CELL 12]: Export Comprehensive Ablation Run Reports, JSON Metrics & Abstract Logs
# ==============================================================================
import os
import json
import time
from datetime import datetime
from collections import Counter, defaultdict
from src.common.audit_logger import (
    classify_10_outcome_str,
    match_gt_label,
    compute_canonical_audit_breakdown,
    format_canonical_audit_table,
    CANONICAL_OUTCOME_CATEGORIES
)
from src.common.label_mapping import canonicalize_label
from src.common.evaluation_framework import evaluate_doc_4_formulations, aggregate_4_formulations

timestamp_str = datetime.now().strftime("%Y%m%d_%H%M%S")
run_folder_name = f"run_{GATING_MODE.lower()}_nov_{W_NOVELTY:.2f}_unc_{W_UNCERTAINTY:.2f}_tau_{THRESHOLD:.2f}_{PROMPT_VERSION.lower()}"
base_export_dir = os.path.join(PROJECT_ROOT, "output/linkner/ablation_reports", CACHE_SPLIT_KEY.replace('/', '_'), run_folder_name)
abstracts_export_dir = os.path.join(base_export_dir, "abstracts")
master_reports_dir = os.path.join(PROJECT_ROOT, "output/linkner/ablation_reports/Master Reports", CACHE_SPLIT_KEY.replace('/', '_'))

os.makedirs(abstracts_export_dir, exist_ok=True)
os.makedirs(master_reports_dir, exist_ok=True)

print("=" * 105)
print(f"  💾 EXPORTING RUN REPORTS & ABSTRACT AUDIT LOGS")
print("=" * 105)
print(f"  • Output Directory : {base_export_dir}")
print(f"  • Total Abstracts  : {len(selected_files)}")

# ------------------------------------------------------------------------------
# 1. Generate Per-Abstract TXT Audit Files
# ------------------------------------------------------------------------------
all_audit_outcomes = []
total_spans_logged = 0
total_escalated_logged = 0

for doc_idx, (txt_path, ann_path, gt_ents) in enumerate(selected_files):
    doc_id = os.path.splitext(os.path.basename(txt_path))[0]
    with open(txt_path, "r", encoding="utf-8") as f:
        raw_text = f.read()
        
    h_preds = all_doc_preds[doc_idx]
    s_preds = spanner_baseline_preds[doc_idx]
    
    lines = []
    lines.append("=" * 100)
    lines.append(f"📄 ABSTRACT AUDIT REPORT: [DOC_ID: {doc_id}]")
    lines.append(f"Run Timestamp: {timestamp_str} | Dataset: {CACHE_SPLIT_KEY} ({len(selected_files)} Abstracts)")
    lines.append(f"Architecture : Single_Call_span_label_passed (Prompt: {PROMPT_VERSION})")
    lines.append(f"Gating Config: {GATING_MODE.upper()} (w_nov={W_NOVELTY:.2f}, w_unc={W_UNCERTAINTY:.2f}, Tau={THRESHOLD:.2f}, Margin Safeguard={MARGIN_THRESHOLD:.2f})")
    lines.append("=" * 100)
    lines.append("")
    
    # 1. Raw Text & Ground Truth
    lines.append("-" * 100)
    lines.append("1. RAW ABSTRACT TEXT & GROUND TRUTH ENTITIES")
    lines.append("-" * 100)
    lines.append("Text:")
    lines.append(f'"{raw_text.strip()}"')
    lines.append("")
    lines.append(f"Ground Truth Entities ({len(gt_ents)} entities):")
    for i, g in enumerate(gt_ents, start=1):
        lbl = canonicalize_label(g.get("label", "O"), use_new_labels=USE_NEW_LABELS)
        lines.append(f"  • [T{i}] ({g.get('start_char', 0)}, {g.get('end_char', 0)}): \"{g.get('text', '')}\" -> {lbl}")
    lines.append("")
    
    # 2. SpanNER Local Neural Inference
    lines.append("-" * 100)
    lines.append(f"2. SPANNER LOCAL NEURAL INFERENCE (Total Candidates: {len(h_preds)})")
    lines.append("-" * 100)
    
    audit_rows = []
    for s_idx, p in enumerate(h_preds, start=1):
        span_t = p.get("span_text", "")
        st_c, end_c = p.get("start_char", 0), p.get("end_char", 0)
        s_lbl = canonicalize_label(p.get("spanner_label", p.get("label", "O")), use_new_labels=USE_NEW_LABELS)
        f_lbl = canonicalize_label(p.get("label", "O"), use_new_labels=USE_NEW_LABELS)
        is_esc = p.get("escalated_to_llm", False)
        u_val = float(p.get("unreliability", p.get("combined", 0.0)))
        m_val = float(p.get("margin", 0.0))
        nov_val = float(p.get("novelty", 0.0))
        unc_val = float(p.get("uncertainty", p.get("u_spanner", 0.0)))
        prob_val = float(p.get("confidence", 0.0))
        
        gt_lbl = match_gt_label(span_t, st_c, end_c, gt_ents)
        outcome = classify_10_outcome_str(gt_label=gt_lbl, spanner_label=s_lbl, final_label=f_lbl, escalated=is_esc)
        all_audit_outcomes.append(outcome)
        
        total_spans_logged += 1
        if is_esc: total_escalated_logged += 1
        
        esc_status = "⚠️ ESCALATE TO LLM" if is_esc else "⚡ KEEP LOCAL"
        lines.append(f"[Span {s_idx}]: '{span_t}' (Chars: {st_c}-{end_c})")
        lines.append(f"  • Top-1 Class   : {s_lbl} (Prob: {prob_val:.4f})")
        lines.append(f"  • Uncertainty   : {unc_val:.4f} ({METHOD.upper()}) | Novelty: {nov_val:.4f} (LOF) | Unreliability: {u_val:.4f}")
        lines.append(f"  • Gating Status : {esc_status} (Unreliability: {u_val:.4f}, Threshold: {THRESHOLD:.2f})")
        if is_esc and p.get("rationale"):
            lines.append(f"  • LLM Rationale : \"{p.get('rationale')}\"")
        lines.append("")
        
        audit_rows.append({
            "span_text": span_t,
            "char_range": f"{st_c}-{end_c}",
            "ground_truth": canonicalize_label(gt_lbl, use_new_labels=USE_NEW_LABELS),
            "spanner_label": s_lbl,
            "hybrid_label": f_lbl,
            "outcome": outcome
        })
        
    # 3. Final Audit Matrix
    lines.append("-" * 100)
    lines.append("3. FINAL AUDIT MATRIX & DECISION CLASSIFICATION")
    lines.append("-" * 100)
    lines.append(f"{'Span Text':<30} | {'Char Range':<12} | {'Ground Truth':<25} | {'SpanNER Label':<25} | {'Final Hybrid':<25} | {'Audit Outcome':<40}")
    lines.append("-" * 175)
    for r in audit_rows:
        lines.append(f"{r['span_text']:<30} | {r['char_range']:<12} | {r['ground_truth']:<25} | {r['spanner_label']:<25} | {r['hybrid_label']:<25} | {r['outcome']:<40}")
    lines.append("")
    
    # Save abstract file
    doc_out_path = os.path.join(abstracts_export_dir, f"{doc_id}.txt")
    with open(doc_out_path, "w", encoding="utf-8") as f:
        f.write("\n".join(lines))

# ------------------------------------------------------------------------------
# 2. Compute Master Metrics Breakdown across All Formulations
# ------------------------------------------------------------------------------
spanner_m = [evaluate_doc_4_formulations(spanner_baseline_preds[i], all_doc_gts[i], use_new_labels=USE_NEW_LABELS) for i in range(len(selected_files))]
hybrid_m = [evaluate_doc_4_formulations(all_doc_preds[i], all_doc_gts[i], use_new_labels=USE_NEW_LABELS) for i in range(len(selected_files))]

spanner_agg = aggregate_4_formulations(spanner_m)
hybrid_agg = aggregate_4_formulations(hybrid_m)
audit_breakdown_10 = compute_canonical_audit_breakdown(all_doc_preds, all_doc_gts, use_new_labels=USE_NEW_LABELS)

# Per-class statistics for strict formulation
class_names = sorted(list(set(g['label'] for gts in all_doc_gts for g in gts if canonicalize_label(g['label']) != 'O')))
per_class_summary = []
for c in class_names:
    c_canon = canonicalize_label(c, use_new_labels=USE_NEW_LABELS)
    s_tp = sum(1 for p, gts in zip(spanner_baseline_preds, all_doc_gts) for item in p if canonicalize_label(item.get('spanner_label', item.get('label', 'O'))) == c_canon and match_gt_label(item['span_text'], item['start_char'], item['end_char'], gts) == c_canon)
    s_pred = sum(1 for p in spanner_baseline_preds for item in p if canonicalize_label(item.get('spanner_label', item.get('label', 'O'))) == c_canon)
    h_tp = sum(1 for p, gts in zip(all_doc_preds, all_doc_gts) for item in p if canonicalize_label(item.get('label', 'O')) == c_canon and match_gt_label(item['span_text'], item['start_char'], item['end_char'], gts) == c_canon)
    h_pred = sum(1 for p in all_doc_preds for item in p if canonicalize_label(item.get('label', 'O')) == c_canon)
    gt_cnt = sum(1 for gts in all_doc_gts for g in gts if canonicalize_label(g['label']) == c_canon)
    
    s_p = s_tp / max(1, s_pred); s_r = s_tp / max(1, gt_cnt); s_f1 = (2*s_p*s_r)/max(1e-9, s_p+s_r)
    h_p = h_tp / max(1, h_pred); h_r = h_tp / max(1, gt_cnt); h_f1 = (2*h_p*h_r)/max(1e-9, h_p+h_r)
    per_class_summary.append({
        "class": c_canon,
        "spanner_f1": s_f1,
        "hybrid_f1": h_f1,
        "delta": h_f1 - s_f1,
        "tp": h_tp,
        "fp": h_pred - h_tp,
        "fn": gt_cnt - h_tp
    })

# ------------------------------------------------------------------------------
# 3. Save run_summary_metrics.json
# ------------------------------------------------------------------------------
summary_metrics_json = {
    "config": {
        "ARCHITECTURE": "Single_Call_span_label_passed",
        "PROMPT_TEMPLATE": PROMPT_VERSION,
        "GATING_MODE": GATING_MODE,
        "W_NOVELTY": W_NOVELTY,
        "W_UNCERTAINTY": W_UNCERTAINTY,
        "METHOD": METHOD,
        "UNRELIABILITY_THRESHOLD": THRESHOLD,
        "MARGIN_THRESHOLD": MARGIN_THRESHOLD,
        "USE_NMS": USE_NMS,
        "DATA_DIR": DATA_DIR,
        "CACHE_SPLIT_KEY": CACHE_SPLIT_KEY
    },
    "total_abstracts": len(selected_files),
    "total_spans": total_spans_logged,
    "escalated_spans": total_escalated_logged,
    "local_computation_saved_pct": (total_spans_logged - total_escalated_logged) / max(1, total_spans_logged) * 100.0,
    "metrics_candidate_arbitration": {
        "hybrid_micro_p": h_mic_p,
        "hybrid_micro_r": h_mic_r,
        "hybrid_micro_f1": h_mic_f1,
        "hybrid_macro_f1": h_mac_f1,
        "spanner_micro_f1": s_mic_f1,
        "spanner_macro_f1": s_mac_f1
    },
    "metrics_3_formulations": {
        "strict_conll": {"spanner": spanner_agg["strict"], "hybrid": hybrid_agg["strict"]},
        "partial_muc7": {"spanner": spanner_agg["partial"], "hybrid": hybrid_agg["partial"]},
        "candidate_mention": {"spanner": spanner_agg["mention"], "hybrid": hybrid_agg["mention"]}
    },
    "canonical_10_outcomes": audit_breakdown_10,
    "per_class": per_class_summary
}

with open(os.path.join(base_export_dir, "run_summary_metrics.json"), "w", encoding="utf-8") as f:
    json.dump(summary_metrics_json, f, indent=4)

# ------------------------------------------------------------------------------
# 4. Save run_summary_report.txt
# ------------------------------------------------------------------------------
report_lines = [
    "=" * 100,
    "📊 LINKNER EXPERIMENT RUN SUMMARY REPORT",
    f"Run Timestamp: {timestamp_str} | Total Abstracts Evaluated: {len(selected_files)}",
    "=" * 100,
    "",
    "----------------------------------------------------------------------------------------------------",
    "1. EXPERIMENT & PIPELINE CONFIGURATION",
    "----------------------------------------------------------------------------------------------------",
    f"  • Architecture              : Single_Call_span_label_passed ({PROMPT_VERSION})",
    f"  • Dataset Directory         : {DATA_DIR} ({len(selected_files)} Abstracts)",
    f"  • Cache Split Key           : {CACHE_SPLIT_KEY}",
    f"  • Label Schema              : {schema_name} (USE_NEW_LABELS = {USE_NEW_LABELS})",
    f"  • Uncertainty Estimator     : {METHOD.upper()} (MCD passes={MCD_PASSES})",
    f"  • Novelty Estimator         : LOF (Precomputed Cache)",
    f"  • Gating Strategy           : {GATING_MODE.upper()} (w_nov={W_NOVELTY:.2f}, w_unc={W_UNCERTAINTY:.2f})",
    f"  • Unreliability Threshold   : {THRESHOLD:.2f}",
    f"  • Margin Safeguard Veto     : {MARGIN_THRESHOLD:.2f}",
    f"  • NMS Policy                : {'ENABLED' if USE_NMS else 'DISABLED (Preserves Raw Candidates)'}",
    f"  • Output Directory          : {base_export_dir}",
    "",
    "----------------------------------------------------------------------------------------------------",
    "2. MULTI-TIER 3-FORMULATION BENCHMARK SUMMARY",
    "----------------------------------------------------------------------------------------------------",
    f"{'Formulation':<25} | {'SpanNER Baseline (P/R/F1)':<28} | {'Hybrid LinkNER (P/R/F1)':<26} | Gain / Delta",
    "-" * 100,
    f"{'Strict CoNLL (1)':<25} | {spanner_agg['strict']['p']*100:>5.1f}% / {spanner_agg['strict']['r']*100:>5.1f}% / {spanner_agg['strict']['f1']*100:>5.1f}% | {hybrid_agg['strict']['p']*100:>5.1f}% / {hybrid_agg['strict']['r']*100:>5.1f}% / {hybrid_agg['strict']['f1']*100:>5.1f}% | {(hybrid_agg['strict']['f1'] - spanner_agg['strict']['f1'])*100:>+6.2f}% F1",
    f"{'Partial MUC-7 (3)':<25} | {spanner_agg['partial']['p']*100:>5.1f}% / {spanner_agg['partial']['r']*100:>5.1f}% / {spanner_agg['partial']['f1']*100:>5.1f}% | {hybrid_agg['partial']['p']*100:>5.1f}% / {hybrid_agg['partial']['r']*100:>5.1f}% / {hybrid_agg['partial']['f1']*100:>5.1f}% | {(hybrid_agg['partial']['f1'] - spanner_agg['partial']['f1'])*100:>+6.2f}% F1",
    f"{'Candidate Mention (4)':<25} | {spanner_agg['mention']['p']*100:>5.1f}% / {spanner_agg['mention']['r']*100:>5.1f}% / {spanner_agg['mention']['f1']*100:>5.1f}% | {hybrid_agg['mention']['p']*100:>5.1f}% / {hybrid_agg['mention']['r']*100:>5.1f}% / {hybrid_agg['mention']['f1']*100:>5.1f}% | {(hybrid_agg['mention']['f1'] - spanner_agg['mention']['f1'])*100:>+6.2f}% F1",
    "",
    "----------------------------------------------------------------------------------------------------",
    "3. PER-CATEGORY PERFORMANCE BREAKDOWN (Strict Matching)",
    "----------------------------------------------------------------------------------------------------",
    f"{'Category':<48} | {'SpanNER F1':<12} | {'Hybrid F1':<12} | {'Gain / Loss':<12} | TP / FP / FN",
    "-" * 105,
]
for c in per_class_summary:
    report_lines.append(f"{c['class']:<48} | {c['spanner_f1']*100:>10.2f}% | {c['hybrid_f1']*100:>10.2f}% | {c['delta']*100:>+10.2f}% | {c['tp']:>3} / {c['fp']:>3} / {c['fn']:>3}")

report_lines.append("")
report_lines.append("----------------------------------------------------------------------------------------------------")
report_lines.append("4. CANONICAL 10-OUTCOME DIAGNOSTIC BREAKDOWN")
report_lines.append("----------------------------------------------------------------------------------------------------")
report_lines.append(format_canonical_audit_table(audit_breakdown_10))

with open(os.path.join(base_export_dir, "run_summary_report.txt"), "w", encoding="utf-8") as f:
    f.write("\n".join(report_lines))

# ------------------------------------------------------------------------------
# 5. Master Reports Index Update
# ------------------------------------------------------------------------------
master_json_path = os.path.join(master_reports_dir, "ablation_summary_metrics.json")
master_data = {}
if os.path.exists(master_json_path):
    try:
        with open(master_json_path, "r", encoding="utf-8") as f:
            master_data = json.load(f)
    except Exception:
        master_data = {}

master_data[run_folder_name] = summary_metrics_json
with open(master_json_path, "w", encoding="utf-8") as f:
    json.dump(master_data, f, indent=4)

print(f"\n✅ Successfully exported:")
print(f"  1. Abstract Reports   : {len(selected_files)} files saved to {abstracts_export_dir}/")
print(f"  2. Run Metrics JSON   : {os.path.join(base_export_dir, 'run_summary_metrics.json')}")
print(f"  3. Summary Report TXT : {os.path.join(base_export_dir, 'run_summary_report.txt')}")
print(f"  4. Master Metrics JSON: {master_json_path}")
print("=" * 105)


  💾 EXPORTING RUN REPORTS & ABSTRACT AUDIT LOGS
  • Output Directory : /Users/hmd/Documents/Workspace/Unified_Test/Ablation/Single_Call_span_label_passed/balanced_val_set/run_prob_or_nov_0.30_unc_0.70_tau_0.35_v1_with_label
  • Total Abstracts  : 80

✅ Successfully exported:
  1. Abstract Reports   : 80 files saved to /Users/hmd/Documents/Workspace/Unified_Test/Ablation/Single_Call_span_label_passed/balanced_val_set/run_prob_or_nov_0.30_unc_0.70_tau_0.35_v1_with_label/abstracts/
  2. Run Metrics JSON   : /Users/hmd/Documents/Workspace/Unified_Test/Ablation/Single_Call_span_label_passed/balanced_val_set/run_prob_or_nov_0.30_unc_0.70_tau_0.35_v1_with_label/run_summary_metrics.json
  3. Summary Report TXT : /Users/hmd/Documents/Workspace/Unified_Test/Ablation/Single_Call_span_label_passed/balanced_val_set/run_prob_or_nov_0.30_unc_0.70_tau_0.35_v1_with_label/run_summary_report.txt
  4. Master Metrics JSON: /Users/hmd/Documents/Workspace/Unified_Test/Ablation/Single_Call_span_label_passed/M